# Population encoding: region-average activity across mice, with LDA1 as a kernel modulator

One model per brain region, **fitted once across all sessions/mice** — not per session.

| | |
|---|---|
| **target** | `y_s(t)` = mean spike count across that region's neurons in session *s*, bin *t*, **z-scored within session** |
| **base block** `B` | the usual Musall design (task event kernels + motor HMM states), each column **z-scored within session** |
| **LDA block** `L` | `z_s · B` — the session's LDA1 score multiplying the base columns, i.e. LDA1 **rescales the encoding kernels** |
| **question** | `ΔR² = cvR²(B + L) − cvR²(B)`, folds held out by **mouse** |
| **null** | shuffle the session → LDA1 assignment (everything *within* a session untouched) |

**Why interactions and not a main effect.** `z_s` is one number per session, so a main-effect
column is constant within a session. The target is z-scored within session (mean 0), so a
within-session constant explains exactly nothing — its least-squares weight is 0. With
within-session normalisation LDA1 can *only* act by changing kernel gains, which is also the
interesting scientific claim ("individuality changes *how* this region encodes the task", not
"high-LDA1 mice have higher firing rates" — that question is already answered in
`firing_rate/fr_psth_ldabin.ipynb`).

**Why held-out mice.** Holding out bins, or sessions of a mouse already in training, leaks
identity: LDA1 is a mouse-identity axis, so the model could memorise the animal. Held-out mice
makes ΔR² answer "does knowing a *new* animal's LDA1 help predict its region activity".

**Why it is fast.** All fits run off per-session sufficient statistics `A_s = B_s'B_s`,
`b_s = B_s'y_s`. Because the LDA block is the base block times a scalar, the Gram matrix for
any set of sessions and *any* LDA assignment is a weighted sum of those with weights
`(1, z_s, z_s²)` — so ~7 M bins are streamed **once**, and every fold and every permutation
afterwards is small linear algebra. See `population_encoding.py` for the derivation.


In [ ]:
import os, importlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import encoding_functions as ef
import population_encoding as pe
importlib.reload(ef); importlib.reload(pe)
warnings.filterwarnings('ignore')

In [ ]:
prefix = '/home/ines/repositories/'
# prefix = '/Users/ineslaranjeira/Documents/Repositories/'

neuron_dir   = prefix + 'representation_learning_variability/paper-individuality/data/neuron_files/'
clustering   = prefix + 'representation_learning_variability/paper-individuality/clustering/data_files/'
results_dir  = 'encoding_results'          # cached per-neuron fits -> used only for the coverage table

LDA_FILE  = 'mouse_LDA_5_bins_cut19-08-2026'
COMPONENT = 0                              # LDA component (0 = LDA1); table columns are 0..23

lda  = pd.read_pickle(clustering + LDA_FILE)
lda1 = pe._lda1(lda, COMPONENT)
print(f'{len(lda1)} sessions with an LDA score  |  component {COMPONENT}  |  '
      f'{lda["mouse_name"].nunique()} mice in the LDA table')

## Which region? (coverage, not intuition)

For a *population-average* target two things matter and they trade off:

* **breadth** — how many sessions/mice have the region at all (drives the LDA1 test's N);
* **depth** — how many neurons per session (a mean over 5 neurons is mostly single-neuron noise,
  which caps `cvR²` for everyone equally but adds variance across sessions).

`region_coverage` reads the cached per-neuron fits, remaps the raw Allen acronyms in the
column names to **Beryl**, pools probes within a session, and drops the coarse/white-matter
labels (`root`, `MB`, `TH`, fibre tracts…) that Beryl assigns to poorly-localised units.

`yield_rho` is the Spearman correlation between neurons-per-session and LDA1: **a region where
that is significant is a trap**, because the number of neurons entering the average would
itself be a function of the predictor.

In [ ]:
cov = pe.region_coverage(results_dir, lda=lda)
cov.head(15)

On the current cache (380 probes, 242 sessions with LDA):

| region | sessions | mice | median neurons/session | ≥5 neurons | ≥10 neurons | yield↔LDA1 |
|---|---|---|---|---|---|---|
| **CA1** | **65** | **42** | 8 | **43 sess / 31 mice** | 29 / 23 | ρ = −0.07, p = 0.59 |
| DG | 60 | 42 | 4 | 24 / 18 | 16 / 14 | ρ = +0.25, p = 0.05 |
| MRN | 48 | 29 | 16 | 42 / 27 | 35 / 24 | ρ = +0.08 |
| CP | 45 | 23 | 47 | 42 / 22 | 37 / 20 | ρ = **−0.42, p = 0.004** ⚠ |
| LP | 39 | 32 | 23 | 32 / 29 | 27 / 26 | ρ = −0.06 |
| PO | 27 | 24 | 36 | 23 / 22 | 21 / 20 | ρ = −0.00 |
| VPM | 21 | 18 | 32 | 18 / 16 | 15 / 13 | ρ = **−0.59, p = 0.005** ⚠ |

**CA1 has the highest session representation** — 65 sessions / 42 mice, more than any other
well-defined Beryl region — so it is the default here.

The cost is **neuron depth**: CA1's median is 8 neurons per session (only 12 sessions reach 20).
The target *is* the mean across those neurons, so with 8 neurons the population trace carries a
lot of Poisson noise. That caps `cvR²` for every session equally — it does **not** bias ΔR² — but
it costs power. `MIN_NEURONS = 5` keeps 43 sessions / 31 mice; `MIN_NEURONS = 10` keeps 29 / 23
with a cleaner target. Both are worth running.

**LP** is the depth-first alternative (39 sessions / 32 mice, median 23 neurons, 27 sess / 26 mice
at ≥10) — fewer sessions, much cleaner per-session traces. Its cache is already built, so
`REGION = 'LP'` runs immediately.

**Avoid CP and VPM** (or control `n_neurons`): their neuron yield tracks LDA1, so the number of
neurons entering the average would itself be a function of the predictor.


In [ ]:
REGION      = 'CA1'     # highest session representation; 'LP' for the depth-first alternative
MIN_NEURONS = 5         # sessions with fewer neurons in REGION are dropped (CA1: 5 -> 43 sessions)
REBIN       = 3         # average this many 16.7 ms bins -> 50 ms (see below); 1 = raw 60 Hz
MOTOR_CONTINUOUS = False  # states-only: avoids task/motor collinearity and the DLC gap bin loss

cache_dir = f'population_stats_{REGION}'

# which probes contain REGION, from the cached per-neuron fits (cheap: no pickles opened)
import glob
res = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(results_dir + '/*.parquet'))],
                ignore_index=True)
res['beryl'] = res['area'].astype(str).map(pe.beryl_map(res['area'].astype(str).unique()))
sessions = (res[res['beryl'].eq(REGION) & res['session'].isin(set(lda1['session']))]
              [['session', 'mouse_name', 'pid']].drop_duplicates()
              .sort_values(['session', 'pid']).reset_index(drop=True))
print(f'{REGION}: {sessions["session"].nunique()} sessions, '
      f'{sessions["mouse_name"].nunique()} mice, {len(sessions)} probes')

## Stream the data once → per-session sufficient statistics

One pass over the binned neuron files. For each session: build the design matrix
(`ef.build_design_matrix`, so the regressors are *identical* to the per-neuron analyses),
average the region's neurons (pooling probes when a session has the region on two of them —
neuron ids restart per probe, so those columns are tagged), **average `REBIN` consecutive
bins**, z-score target and columns within session, and store `A = B'B`, `b = B'y`, `y'y`, `n`.

**Why rebin.** At 60 Hz the mean spike count of ~10 neurons is dominated by Poisson noise, which
caps the achievable R² for everyone equally. Averaging to 50 ms cuts that white noise by √3 and
leaves the kernels (all ≥ 0.3 s wide) intact — on LP it lifted base `cvR²` from 0.017 to 0.061,
i.e. 3.6× more signal to detect an LDA effect in. Blocks are formed only *within* runs of
contiguous kept bins, so nothing is averaged across a peri-trial mask gap.

Cached one pickle per session in `population_stats_<REGION>/` — resumable, and the fitting below
never touches the raw data again (~1.7 s per session).


In [ ]:
stats = pe.sweep_sessions(sessions, neuron_dir, REGION, cache_dir,
                          motor_continuous=MOTOR_CONTINUOUS, min_neurons=MIN_NEURONS,
                          rebin=REBIN)
S = pe.assemble(stats)
print(f'\n{len(S["session"])} sessions | {len(np.unique(S["mouse"]))} mice | '
      f'{S["n"].sum():,} bins | {len(S["cols"])} base columns')
print(pd.Series(S['groups']).value_counts().to_string())
print(f'neurons/session: median {np.median(S["n_neurons"]):.0f} '
      f'(min {S["n_neurons"].min()}, max {S["n_neurons"].max()})')

### The regressors (identical to the per-neuron encoding analyses)

From `encoding_functions.DEFAULT_KERNELS` / `MOTOR_STATE_COLS` — raised-cosine kernels,
windows in seconds relative to the aligning event:

**TASK** (event kernels)
* `stimOn` 0 → 0.6 s, 6 bases — aligned to `goCueTrigger_times`
* `fb_correct` 0 → 1.0 s, 8 bases · `fb_error` 0 → 1.0 s, 8 bases — aligned to feedback
* `choice` −0.25 → 0.75 s, 8 bases, amplitude ±1 — aligned to **firstMovement** (not stimOn: RTs
  vary, so a stimOn-locked choice kernel smears across the RT)
* `prior` 0 → 1.0 s, 6 bases, amplitude `block − 0.5` (−0.3 / 0 / +0.3) — stimOn-aligned
* `scontrast` 0 → 0.6 s, 6 bases, signed contrast (+right / −left) — stimOn-aligned

**MOTOR STATES** — `paw` / `whisk` / `lick` discrete HMM states, one-hot (one level dropped),
each with a ±150 ms raised-cosine lag basis (4 bases). NaN state → bin dropped, never folded
into the reference level.

**MOTOR CONTINUOUS** — off by default (`MOTOR_CONTINUOUS = False`). Turn on for L/R paw speed
(log1p), whisker motion energy, lick count (log1p); costs bins to DLC tracking gaps (never
interpolated) and adds task/motor collinearity.

Bins are restricted to `[stimOn − 0.5 s, feedback + 2 s]` and any bin with a NaN regressor is
dropped.

## Does LDA1 help predict held-out mice?

`cvR²` is measured against each session's own mean (target is z-scored within session), so it is
"variance of the region-average trace explained by task + behaviour". Three LDA blocks:

* `all` — LDA1 × every base column
* `task` — LDA1 × task kernels only
* `motor` — LDA1 × motor-state columns only

In [ ]:
folds = pe.mouse_folds(S, n_splits=5)
print('fold sizes (sessions / mice held out):',
      [(len(te), len(np.unique(S['mouse'][te]))) for _, te in folds])

z = pe.lda_vector(S, lda1, level='session')     # z-scored LDA1 per session
base = pe.cv_r2(S, z, np.array([], int), folds)
print(f'\nbase model (task + motor states):  cvR2 = {base["cv_r2"]:.4f}   '
      f'per-fold {np.round(base["r2_folds"], 3)}')

rows = []
for which in ['all', 'task', 'motor']:
    d = pe.delta_r2(S, z, folds, which=which, base=base)
    rows.append(dict(block=which, n_cols=d['n_interactions'],
                     cv_r2_base=d['cv_r2_base'], cv_r2_full=d['cv_r2_full'], dR2=d['dR2']))
    globals()[f'obs_{which}'] = d
pd.DataFrame(rows).round(5)

## Permutation null: shuffle the session → LDA1 assignment

This is the null you asked for. Because the LDA score is **one number per session**, permuting
*within* a session is a no-op — what has to be broken is the session↔LDA link, and shuffling
across sessions leaves everything within a session (design, target, neuron pooling, bin count)
untouched by construction. Two levels:

* **`session`** — shuffle `z` across sessions.
* **`mouse`** — shuffle the per-mouse value across mice, all sessions of a mouse keeping one
  value. This is the **headline test**: it matches the held-out-mouse CV and cannot be won by
  within-mouse session structure. With ~1 session per mouse in LP the two are nearly identical;
  report both.

Fold alphas are frozen at the observed-data values (re-selecting per shuffle would be circular),
and the base model does not involve `z` at all, so its `cvR²` is computed once.

In [ ]:
N_PERM = 2000

nulls = {}
for which in ['all', 'task', 'motor']:
    for level in ['session', 'mouse']:
        zl = pe.lda_vector(S, lda1, level=level)
        obs = pe.delta_r2(S, zl, folds, which=which, base=base) if level == 'mouse' \
              else globals()[f'obs_{which}']
        nulls[(which, level)] = pe.perm_null(S, zl, folds, which=which, level=level,
                                             n_perm=N_PERM, observed=obs)
        r = nulls[(which, level)]
        print(f'{which:6s} × LDA1  [{level:7s} null]  dR2={r["dR2"]:+.5f}  '
              f'null {r["null_mean"]:+.5f}±{r["null_sd"]:.5f}  z={r["z"]:+.2f}  p={r["p"]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for j, which in enumerate(['all', 'task', 'motor']):
    for i, level in enumerate(['session', 'mouse']):
        pe.plot_null(nulls[(which, level)], ax=axes[i, j])
plt.tight_layout(); plt.show()

### Reading the result

* `ΔR²` **> 0 and above the null** → LDA1 carries information about how this region encodes
  task/behaviour that generalises to unseen animals.
* `ΔR²` **≈ 0 or negative** → the interaction block only costs degrees of freedom. Note the null
  is centred *below* zero: adding 40–80 useless columns hurts held-out R², so the null mean is
  the right reference point, not 0.
* A significant `session`-level but non-significant `mouse`-level result means the effect lives
  in within-mouse session-to-session variation, not in animal identity — worth knowing, but it
  is not an individuality claim.

## Where does the gain sit? (descriptive)

Fitted on all sessions at the median fold alpha. `w_lda` is how much LDA1 rescales each base
regressor: positive = higher-LDA1 animals have a *larger* gain on that regressor. Significance
comes from the permutation above, not from these weights.

In [ ]:
W = pe.interaction_weights(S, z, which='all', folds=folds)
fam = (W.groupby(['group', 'family'])['w_lda']
         .agg(mean_w='mean', max_abs=lambda s: s.abs().max(), n='size')
         .reset_index().sort_values('max_abs', ascending=False))

fig, ax = plt.subplots(figsize=(7, 4.5))
colors = {'task': '#3a7ca5', 'motor_states': '#c1442a', 'motor_continuous': '#6a9a4f'}
ax.barh(fam['family'], fam['mean_w'], color=[colors.get(g, 'grey') for g in fam['group']])
ax.axvline(0, color='k', lw=.8)
ax.set(xlabel='mean LDA1 × regressor weight', title=f'{REGION}: how LDA1 rescales each kernel family')
ax.invert_yaxis(); plt.tight_layout(); plt.show()
fam.round(4)

## Caveats

1. **Depth caps everything.** With a median of ~23 neurons (LP) the region average still carries
   single-neuron noise, so `cvR²` is an underestimate of the true population encoding. It biases
   *all* sessions the same way, so it does not manufacture an LDA1 effect — but it does cost power.
2. **`n_neurons` is not controlled.** LP was chosen precisely because yield does not track LDA1
   (ρ = −0.06). If you switch to CP or VPM, add `n_neurons` (or subsample to a fixed neuron count
   per session) before believing any ΔR².
3. **Per-session z-scoring of the design** is what makes sessions comparable and kills the LDA
   main effect. It also means each session contributes equally per bin, so long sessions still
   dominate by bin count — switch to per-session weights in `_gram` if that matters.
4. **One component at a time.** `COMPONENT = 0` is LDA1. Running components 0..23 needs FDR
   correction across components; do not fish.
5. **Not the same cvR² as `fit_session(unit='region')`** — that fits each session separately with
   its own weights. Here the weights are shared across animals, so `cvR²` is lower by
   construction and the two numbers are not comparable.